# 有限長円形導波管の解析的な $S$ パラメータ

一様な円形導波管を理想的に整合された2ポートとして扱い、支配モード $\mathrm{TE}_{11}$ のカットオフ周波数と、有限長区間を伝搬する際の $S_{21}$ の位相を解析的に求めます。

この理想モデルでは

$$S_{11}=0,\qquad |S_{21}|=1,\qquad S_{21}=e^{-j\beta L}$$

とし、時間依存性と伝搬方向には $e^{j\omega t-j\beta z}$ を採用します。伝搬定数は

$$f_c=\frac{x'_{11}c}{\pi D},\qquad
\beta(f)=\frac{2\pi f}{c}\sqrt{1-\left(\frac{f_c}{f}\right)^2}$$

です。ここで $x'_{11}=1.84118\ldots$ は $J_1'(x)$ の第1零点、$D$ は導波管直径、$L$ は導波管長です。

> **適用範囲:** $|S_{21}|=1$ は損失・不連続・ポート反射・高次モードを無視した単一モードの理想モデルです。$f\le f_c$ では $\beta$ が実数でなくなるため、下の位相計算は伝搬帯域 ($f>f_c$) のみを対象とします。

## 計算条件

直径と掃引周波数は指定値です。位相は長さに依存するため、`length_mm` を任意に変更できるパラメータとし、初期値を 10 mm にしています。`n_points` も必要に応じて変更できます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Geometry and sweep parameters
diameter_mm = 2.26
length_mm = 10.0       # 任意の有限長に変更可能
f_start_GHz = 90.0
f_stop_GHz = 115.0
n_points = 501

# SI units
diameter = diameter_mm * 1e-3
length = length_mm * 1e-3
frequencies = np.linspace(f_start_GHz, f_stop_GHz, n_points) * 1e9

C0 = 299_792_458.0
X11_PRIME = 1.8411837813406593

## カットオフ周波数と $S$ パラメータ

In [ ]:
cutoff_frequency = X11_PRIME * C0 / (np.pi * diameter)

if np.any(frequencies <= cutoff_frequency):
    raise ValueError(
        "位相を計算する全周波数は TE11 カットオフ周波数より高くしてください。"
    )

k0 = 2 * np.pi * frequencies / C0
beta = k0 * np.sqrt(1 - (cutoff_frequency / frequencies) ** 2)
s11 = np.zeros_like(frequencies, dtype=complex)
s21 = np.exp(-1j * beta * length)
phase_wrapped_deg = np.angle(s21, deg=True)
phase_unwrapped_deg = np.rad2deg(np.unwrap(np.angle(s21)))

print(f"直径                  : {diameter_mm:.3f} mm")
print(f"導波管長              : {length_mm:.3f} mm")
print(f"TE11 カットオフ周波数 : {cutoff_frequency / 1e9:.6f} GHz")
print(f"掃引範囲              : {frequencies[0] / 1e9:.1f}–{frequencies[-1] / 1e9:.1f} GHz")
print(f"S21 位相（unwrap）     : {phase_unwrapped_deg[0]:.3f}° → {phase_unwrapped_deg[-1]:.3f}°")

`phase_wrapped_deg` は通常の $[-180^\circ,180^\circ]$ 表示、`phase_unwrapped_deg` は周波数に対して連続になるようにアンラップした表示です。絶対位相は基準面と `length_mm` に依存します。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
f_GHz = frequencies / 1e9

axes[0].plot(f_GHz, np.abs(s11), label=r"$|S_{11}|$")
axes[0].plot(f_GHz, np.abs(s21), label=r"$|S_{21}|$")
axes[0].set_ylabel("Magnitude")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(f_GHz, phase_wrapped_deg, label="Wrapped")
axes[1].plot(f_GHz, phase_unwrapped_deg, "--", label="Unwrapped")
axes[1].set_xlabel("Frequency (GHz)")
axes[1].set_ylabel(r"Phase of $S_{21}$ (deg)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle(
    f"Ideal circular waveguide: D={diameter_mm:g} mm, L={length_mm:g} mm"
)
fig.tight_layout()
plt.show()

## 数値チェック

In [ ]:
# 理想モデルの前提と、解析式をコードが満たすことを確認
assert cutoff_frequency < frequencies[0]
assert np.all(s11 == 0)
assert np.allclose(np.abs(s21), 1.0, rtol=0, atol=1e-14)
assert np.allclose(phase_unwrapped_deg, -np.rad2deg(beta * length), atol=1e-10)
print("Checks passed: sweep is above cutoff, S11=0, |S21|=1, phase=-beta*L.")

## 別の寸法で使う方法

`diameter_mm` と `length_mm` を変更して上から再実行すれば、任意の直径・有限長の一様円形導波管を計算できます。直径を変更すると $f_c$ と $\beta$ の両方が、長さだけを変更すると位相 $-\beta L$ が変化します。掃引にカットオフ以下が含まれる場合は、理想的な単位振幅伝送という前提が成立しないため明示的にエラーにしています。